In [1]:
%%file generuj_dane.py
import csv, random

random.seed(44)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def czy_to_fraud(kwota, godzina, tx_ostatnia_godz):
    flagi = 0
    if kwota > 4500: flagi += 1
    if 0 <= godzina <= 2: flagi += 1   # Krótsza noc (tylko 0, 1, 2)
    if tx_ostatnia_godz >= 5: flagi += 1 # Rzadsza seria
    
    if flagi >= 2: return random.random() < 0.85
    elif flagi == 1: return random.random() < 0.05
    else: return random.random() < 0.001

wiersze = []
# Generujemy 10 000 transakcji, żeby model miał na czym się uczyć
for _ in range(10000):
    kwota = round(random.uniform(5.0, 5000.0), 2)
    godzina = random.randint(0, 23)
    tx_ostatnia_godz = random.choices([0, 1, 2, 3, 4, 5, 6, 7], weights=[35, 25, 15, 10, 6, 4, 3, 2])[0]

    sklep = random.choice(sklepy)
    kategoria = random.choice(kategorie)
    fraud = czy_to_fraud(kwota, godzina, tx_ostatnia_godz)

    wiersze.append({
        'kwota': kwota,
        'sklep': sklep,
        'kategoria': kategoria,
        'godzina': godzina,
        'tx_ostatnia_godz': tx_ostatnia_godz,
        'czy_fraud': int(fraud),
    })

with open('dane.csv', 'w', newline='', encoding='utf-8') as f:
    pola = ['kwota', 'sklep', 'kategoria', 'godzina', 'tx_ostatnia_godz', 'czy_fraud']
    w = csv.DictWriter(f, fieldnames=pola)
    w.writeheader()
    w.writerows(wiersze)

n_fraud = sum(r['czy_fraud'] for r in wiersze)
print(f"Zapisano {len(wiersze)} transakcji.")
print(f"Fraudów: {n_fraud} ({n_fraud/len(wiersze)*100:.1f}%)")

Overwriting generuj_dane.py


In [2]:
%%file trenuj_model.py
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# 1. Wczytanie wygenerowanych danych
df = pd.read_csv('dane.csv')

# 2. Kodowanie tekstu na liczby
le_sklep = LabelEncoder()
le_kategoria = LabelEncoder()
df['sklep_kod'] = le_sklep.fit_transform(df['sklep'])
df['kategoria_kod'] = le_kategoria.fit_transform(df['kategoria'])

cechy = ['kwota', 'sklep_kod', 'kategoria_kod', 'godzina', 'tx_ostatnia_godz']
X = df[cechy]
y = df['czy_fraud']

# 3. Podział z zachowaniem proporcji klas (stratify=y) - KLUCZOWE przy 2% fraudów
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=44, stratify=y
)

# 4. Trening modelu ze zbalansowanymi wagami klas i płytkimi drzewami
model = RandomForestClassifier(
    max_depth=8,                
    class_weight='balanced',    
    random_state=44
)
model.fit(X_train, y_train)

# 5. Ocena skuteczności
y_pred = model.predict(X_test)
print("=== RAPORT KLASYFIKACJI ===")
print(classification_report(y_test, y_pred, target_names=['normalna', 'FRAUD']))

print("=== MACIERZ POMYŁEK ===")
cm = confusion_matrix(y_test, y_pred)
print(f"                przewid. norm.  przewid. fraud")
print(f"prawdziwa norm.     {cm[0][0]:>8}       {cm[0][1]:>8}")
print(f"prawdziwy fraud     {cm[1][0]:>8}       {cm[1][1]:>8}")

# 6. Zapis modelu i enkoderów do pliku
with open('model.pkl', 'wb') as f:
    pickle.dump({
        'model': model,
        'le_sklep': le_sklep,
        'le_kategoria': le_kategoria,
        'cechy': cechy,
    }, f)
print("\nZapisano model do pliku model.pkl")

Overwriting trenuj_model.py


In [3]:
%%file producent.py
from kafka import KafkaProducer
from collections import defaultdict
import json, random, time
from datetime import datetime

producent = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

historia = defaultdict(list)

def czy_to_fraud(kwota, godzina, tx_ostatnia_godz):
    flagi = 0
    if kwota > 4500: flagi += 1
    if 0 <= godzina <= 2: flagi += 1
    if tx_ostatnia_godz >= 5: flagi += 1
    
    if flagi >= 2: return random.random() < 0.85
    elif flagi == 1: return random.random() < 0.05
    else: return random.random() < 0.001

for i in range(2000):
    teraz = time.time()
    
    kid = f'k{random.randint(1, 2500):04d}'

    historia[kid] = [t for t in historia[kid] if teraz - t <= 3600]
    tx_ostatnia_godz = len(historia[kid])

    kwota = round(random.uniform(5.0, 5000.0), 2)
    godzina = random.randint(0, 23)
    sklep = random.choice(sklepy)
    kategoria = random.choice(kategorie)
    
    fraud = czy_to_fraud(kwota, godzina, tx_ostatnia_godz)

    tx = {
        'id_tx': f'TX{random.randint(1000,9999)}',
        'id_klienta': kid,
        'kwota': kwota,
        'sklep': sklep,
        'kategoria': kategoria,
        'godzina': godzina,
        'tx_ostatnia_godz': tx_ostatnia_godz,
        'czy_fraud': bool(fraud),
        'czas': datetime.now().isoformat(),
    }
    producent.send('transakcje', value=tx)
    historia[kid].append(teraz)

    tag = "FRAUD" if fraud else "ok"
    print(f"[{i+1}] {tx['id_tx']} | {kid} | {kwota:>7.2f} PLN | {sklep:<9} | "
          f"g={godzina:02d} | {tx_ostatnia_godz}tx/h | {tag}")
    time.sleep(0.5)

producent.flush()
producent.close()

Overwriting producent.py


In [4]:
%%file api.py
from flask import Flask, request, jsonify
import pickle
import pandas as pd

app = Flask(__name__)

# Wczytaj model raz, na starcie serwera (pickle z Lab 7)
with open('model.pkl', 'rb') as f:
    paczka = pickle.load(f)
model = paczka['model']
le_sklep = paczka['le_sklep']
le_kategoria = paczka['le_kategoria']
cechy = paczka['cechy']

def decyzja(p):
    if p > 0.7:
        return "BLOKUJ"
    elif p >= 0.35:
        return "WERYFIKUJ"
    return "ZATWIERDŹ"

@app.route("/ocena", methods=["POST"])
def ocena():
    tx = request.get_json()
    if not tx or "kwota" not in tx:
        return jsonify({"blad": "Brak pola 'kwota'"}), 400
    try:
        sklep_kod = le_sklep.transform([tx['sklep']])[0]
        kategoria_kod = le_kategoria.transform([tx['kategoria']])[0]
    except (ValueError, KeyError):
        return jsonify({"blad": "Nieznany sklep lub kategoria"}), 400

    wiersz = pd.DataFrame([{
        'kwota': tx['kwota'],
        'sklep_kod': sklep_kod,
        'kategoria_kod': kategoria_kod,
        'godzina': tx.get('godzina', 12),
        'tx_ostatnia_godz': tx.get('tx_ostatnia_godz', 0),
    }])[cechy]

    p = float(model.predict_proba(wiersz)[0][1])   # prawdopodobieństwo fraudu
    return jsonify({
        "id_tx": tx.get("id_tx", "brak"),
        "prawdopodobienstwo": round(p, 3),
        "decyzja": decyzja(p),
    })

@app.route("/zdrowie")
def zdrowie():
    return jsonify({"status": "ok", "cechy_modelu": cechy})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting api.py


In [5]:
%%file scorer.py
from kafka import KafkaConsumer, KafkaProducer
import json, requests

konsument = KafkaConsumer(
    'transakcje',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    group_id='scorer',
)
producent = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

print("Konsument oceniający uruchomiony — czekam na transakcje...")

for wiadomosc in konsument:
    tx = wiadomosc.value
    try:
        r = requests.post("http://localhost:5000/ocena", json=tx, timeout=5)
        wynik = r.json()
    except Exception as e:
        print(f"Błąd połączenia z API: {e}")
        continue
    if "decyzja" not in wynik:
        print(f"API zwróciło błąd: {wynik}")
        continue

    zdarzenie = {
        'id_tx': tx['id_tx'],
        'id_klienta': tx['id_klienta'],
        'kwota': tx['kwota'],
        'sklep': tx['sklep'],
        'decyzja': wynik['decyzja'],
        'prawdopodobienstwo': wynik['prawdopodobienstwo'],
        'czy_fraud': tx.get('czy_fraud', False),
    }
    producent.send('decyzje', value=zdarzenie)

    znacznik = "  <-- FRAUD!" if tx.get('czy_fraud') else ""
    print(f"{tx['id_tx']} | {tx['kwota']:>7.2f} PLN | "
          f"{wynik['decyzja']:<10} (p={wynik['prawdopodobienstwo']}){znacznik}")

Overwriting scorer.py


In [6]:
%%file monitoring.py
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType
from pyspark.sql.functions import from_json, col, window, count, sum as _sum, when, round as _round

spark = SparkSession.builder.appName("MonitoringFraud").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

schema = StructType([
    StructField("id_tx", StringType()),
    StructField("id_klienta", StringType()),
    StructField("kwota", DoubleType()),
    StructField("sklep", StringType()),
    StructField("decyzja", StringType()),
    StructField("prawdopodobienstwo", DoubleType()),
    StructField("czy_fraud", BooleanType()),
])

kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "decyzje")
    .load()
)

df = (
    kafka_raw
    .select(
        from_json(col("value").cast("string"), schema).alias("d"),
        col("timestamp").alias("czas_kafka"),
    )
    .select("d.*", "czas_kafka")
)

# Zwiększono okno do 5 minut i watermarking do 1 minuty dla płynniejszej agregacji testowej
okna = (
    df
    .withWatermark("czas_kafka", "1 minute")
    .groupBy(window("czas_kafka", "5 minutes", "1 minute")) # Okno 5 min, przesuwa się co 1 min
    .agg(
        count("id_tx").alias("sprawdzono"),
        _sum(when(col("decyzja") == "BLOKUJ", 1).otherwise(0)).alias("zablokowano"),
        _round(_sum(when(col("decyzja") == "BLOKUJ", col("kwota")).otherwise(0.0)), 2).alias("oszczedzono_PLN"),
        _sum(when(col("decyzja") == "WERYFIKUJ", 1).otherwise(0)).alias("do_weryfikacji"),
    )
)

zapytanie = (
    okna.writeStream
    .format("console")
    .outputMode("update") # Zmiana z complete na update - lepsze do strumieni z oknami
    .option("truncate", False)
    .start()
)
zapytanie.awaitTermination()


Overwriting monitoring.py


In [8]:
%%file dashboard.py
from kafka import KafkaConsumer
from collections import Counter
import json

konsument = KafkaConsumer(
    'decyzje',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    group_id='dashboard',
)

decyzje = Counter()
oszczedzono = 0.0
sprawdzono = 0
trafione = 0       # zablokowane i faktycznie fraud
przepuszczone = 0  # niezablokowane, a były fraudem

print("Panel antyfraudowy uruchomiony...")

for wiadomosc in konsument:
    d = wiadomosc.value
    sprawdzono += 1
    decyzje[d['decyzja']] += 1
    zablokowana = d['decyzja'] == 'BLOKUJ'
    fraud = d.get('czy_fraud', False)

    if zablokowana:
        oszczedzono += d['kwota']
        if fraud:
            trafione += 1
    elif fraud:
        przepuszczone += 1

    if sprawdzono % 10 == 0:
        skutecznosc = trafione / (trafione + przepuszczone) * 100 if (trafione + przepuszczone) else 0
        print("\n" + "=" * 42)
        print("           PANEL ANTYFRAUDOWY")
        print("=" * 42)
        print(f"Sprawdzono transakcji:  {sprawdzono}")
        print(f"Zablokowano:            {decyzje['BLOKUJ']}")
        print(f"Do weryfikacji:         {decyzje['WERYFIKUJ']}")
        print(f"Zatwierdzono:           {decyzje['ZATWIERDŹ']}")
        print(f"Oszczędzono:            {oszczedzono:,.2f} PLN")
        print(f"Wykryto fraudów:        {skutecznosc:.0f}%  (czułość)")
        print(f"Przepuszczone fraudy:   {przepuszczone}")
        print("=" * 42 + "\n")

Overwriting dashboard.py
